
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>





# Extract Data Lab

In this lab, you will extract raw data from JSON files.

## Learning Objectives
By the end of this lab, you should be able to:
- Register an external table to extract data from JSON files




## Run Setup

Run the following cell to configure variables and datasets for this lesson.

In [0]:
%run ./Includes/Classroom-Setup-02.3L




## Overview of the Data

We will work with a sample of raw Kafka data written as JSON files. 

Each file contains all records consumed during a 5-second interval, stored with the full Kafka schema as a multiple-record JSON file. 

The schema for the table:

| field  | type | description |
| ------ | ---- | ----------- |
| key    | BINARY | The **`user_id`** field is used as the key; this is a unique alphanumeric field that corresponds to session/cookie information |
| offset | LONG | This is a unique value, monotonically increasing for each partition |
| partition | INTEGER | Our current Kafka implementation uses only 2 partitions (0 and 1) |
| timestamp | LONG    | This timestamp is recorded as milliseconds since epoch, and represents the time at which the producer appends a record to a partition |
| topic | STRING | While the Kafka service hosts multiple topics, only those records from the **`clickstream`** topic are included here |
| value | BINARY | This is the full data payload (to be discussed later), sent as JSON |



 
## Extract Raw Events From JSON Files
To load this data into Delta properly, we first need to extract the JSON data using the correct schema.

Create an external table against JSON files located at the filepath provided below. Name this table **`events_json`** and declare the schema above.

Hint: Make sure you:
1. Use a CTAS statement
2. Use CAST to ensure the data types are correct.

In [0]:
CREATE OR REPLACE TABLE events_json
AS
SELECT
  CAST(key AS BINARY) AS key,
  CAST(offset AS BIGINT) AS offset,
  CAST(partition AS INT) AS partition,
  CAST(timestamp AS BIGINT) AS timestamp,
  CAST(topic AS STRING) AS topic,
  CAST(value AS BINARY) AS value
FROM json.`${DA.paths.kafka_events}`




**NOTE**: We'll use Python to run checks occasionally throughout the lab. The following cell will return an error with a message on what needs to change if you have not followed instructions. No output from cell execution means that you have completed this step.

In [0]:
%python
assert spark.table("events_json"), "Table named `events_json` does not exist"
assert spark.table("events_json").columns == ['key', 'offset', 'partition', 'timestamp', 'topic', 'value'], "Please name the columns in the order provided above"
assert spark.table("events_json").dtypes == [('key', 'binary'), ('offset', 'bigint'), ('partition', 'int'), ('timestamp', 'bigint'), ('topic', 'string'), ('value', 'binary')], "Please make sure the column types are identical to those provided above"

total = spark.table("events_json").count()
assert total == 2252, f"Expected 2252 records, found {total}"



 
Run the following cell to delete the tables and files associated with this lesson.

In [0]:
%python
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>